<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/28_interpolation/20_cubic_spline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)


In [ ]:
# 그래프, 수학 기능 추가
# Add graph and math features
import matplotlib.pyplot as plt
import numpy as np
import scipy.interpolate as si


# 3차 스플라인 내삽<br>Cubic Spline Interpolation


앞에서 보았듯, *전역* 다항식의 차수를 무작정 올리면 양 끝에서 폭발한다 (룽게 현상). 반대 방향으로 가 보면 어떨까? &mdash; 구간을 잘게 나누고, 각 조각마다 *낮은* 차수의 다항식을 쓰자.<br>
The previous notebook showed that cranking up the degree of a *global* polynomial blows up near the endpoints (the Runge phenomenon). A natural alternative is the opposite strategy &mdash; chop the interval into small pieces and fit a *low*-degree polynomial on each.

이렇게 만들어진 구간별 다항식을 **스플라인** 이라 한다. 스플라인이라는 이름은 제도사가 자유 곡선을 그릴 때 사용하던, 무게추로 누른 얇은 나무 자에서 왔다 &mdash; 자가 자연스럽게 만드는 곡선이 곧 자연 3차 스플라인이다.<br>
Such piecewise polynomials are called **splines**. The name comes from the thin wooden ruler that draftsmen pressed into a smooth curve with weights &mdash; the curve such a ruler naturally takes is *exactly* the natural cubic spline.


## 구간별 다항식<br>Piecewise Polynomials


$n+1$ 개의 자료점 $(x_0, y_0), \ldots, (x_n, y_n)$ 를 정렬된 절점이라 하자. **3차 스플라인** $S(x)$ 는 각 구간 $[x_i, x_{i+1}]$ 위에서 3차 다항식 $S_i(x)$ 가 되는 함수이다.<br>
Let $(x_0, y_0), \ldots, (x_n, y_n)$ be data points with sorted $x$ coordinates. A **cubic spline** $S(x)$ is a function that is a cubic polynomial $S_i(x)$ on each subinterval $[x_i, x_{i+1}]$:

$$
S_i(x) \;=\; a_i + b_i (x - x_i) + c_i (x - x_i)^2 + d_i (x - x_i)^3,
\qquad x \in [x_i, x_{i+1}].
$$

각 조각의 차수는 작지만 (3차), 조각 *수* 가 자료점만큼 늘어나므로 표현력은 충분히 풍부하다. 게다가 각 조각이 짧으므로, 등간격 라그랑주에서 보았던 르베그 상수 폭발이 일어나지 않는다.<br>
Each piece has low degree (3), but the *number* of pieces grows with the data, giving plenty of expressive power. And because each piece is short, the Lebesgue-constant blow-up that plagued equispaced Lagrange does not occur.


## 연속성 조건<br>Continuity Conditions


각 조각이 자기 양 끝의 자료값을 지나는 것만으로는 부족하다 &mdash; 조각과 조각 사이가 *부드럽게* 이어져야 한다.<br>
Just having each piece pass through its endpoints is not enough &mdash; the joins between pieces must also be *smooth*.

스플라인의 부드러움은 다음과 같은 *연속성 조건* 으로 표현한다.<br>
Smoothness is expressed as a chain of *continuity conditions* at every interior knot $x_i$:

| 조건<br>Condition | 의미<br>Meaning | 시각적 효과<br>Visual effect |
|---|---|---|
| $C^0$ &mdash; $S_i(x_i) = S_{i-1}(x_i)$ | 함수값이 연속<br>Function value continuous | 끊어지지 않음 / no jumps |
| $C^1$ &mdash; $S_i'(x_i) = S_{i-1}'(x_i)$ | 1차 미분이 연속<br>Slope continuous | 꺾이지 않음 / no kinks |
| $C^2$ &mdash; $S_i''(x_i) = S_{i-1}''(x_i)$ | 2차 미분이 연속<br>Curvature continuous | 곡률이 매끄럽게 변함 / smooth curvature |

3차 스플라인은 일반적으로 $C^2$ 를 요구한다. 미지수 개수와 조건 개수를 세어 보자.<br>
A cubic spline is normally required to be $C^2$. Let&rsquo;s count unknowns and conditions:

* 미지수: 조각마다 4개 ($a_i, b_i, c_i, d_i$), 조각 수 $n$ &rarr; $4n$ 개. / Unknowns: 4 per piece, $n$ pieces &rArr; $4n$.
* 보간 조건: $2n$ 개 (각 조각이 양 끝의 자료점을 지남). / Interpolation: $2n$ (each piece pinned at both ends).
* 내부 절점에서 $C^1$ 연속: $n - 1$ 개. / Interior $C^1$: $n - 1$.
* 내부 절점에서 $C^2$ 연속: $n - 1$ 개. / Interior $C^2$: $n - 1$.
* 합계: $2n + (n - 1) + (n - 1) = 4n - 2$.

따라서 $4n$ 개의 미지수에 대해 식이 $4n - 2$ 개. 식이 두 개 부족하다 &mdash; 양 끝의 *경계 조건* 이 필요하다.<br>
That gives $4n - 2$ equations for $4n$ unknowns &mdash; we are short by **two**. The missing equations are the *boundary conditions* at the two ends of the data.


**시각적으로 살펴보기.** 같은 5개 자료점을 (a) 직선으로 잇기 ($C^0$ 만), (b) 1차 미분만 맞추는 곡선 ($C^1$), (c) 자연 3차 스플라인 ($C^2$) 으로 비교해 보자.<br>
**A visual comparison.** The same five data points joined three ways: (a) straight lines (only $C^0$), (b) a curve that matches first derivatives ($C^1$), (c) a natural cubic spline ($C^2$).


In [ ]:
x_pts = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_pts = np.array([0.0, 0.8, 0.9, 0.1, -0.6])

x_dense = np.linspace(x_pts[0], x_pts[-1], 400)

# (a) C^0 only: piecewise linear / 구간별 직선
y_linear = np.interp(x_dense, x_pts, y_pts)

# (b) C^1: PCHIP (shape-preserving piecewise cubic Hermite)
#     PCHIP 는 1차 미분만 맞추고 2차 미분은 일반적으로 불연속.
pchip = si.PchipInterpolator(x_pts, y_pts)
y_pchip = pchip(x_dense)

# (c) C^2: natural cubic spline / 자연 3차 스플라인
spline = si.CubicSpline(x_pts, y_pts, bc_type='natural')
y_spline = spline(x_dense)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, y, label in zip(
        axes,
        (y_linear, y_pchip, y_spline),
        ('(a) $C^0$: piecewise linear', '(b) $C^1$: PCHIP', '(c) $C^2$: natural cubic spline'),
):
    ax.plot(x_dense, y)
    ax.plot(x_pts, y_pts, 'ko')
    ax.set_title(label)
    ax.grid(True)
    ax.set_xlabel('$x$')
axes[0].set_ylabel('$y$')
plt.tight_layout()
plt.show()


(a) 는 절점에서 *꺾이고*, (b) 는 매끄럽게 이어지지만 곡률이 불연속이며, (c) 는 곡률까지 매끄럽게 이어진다. 이 차이는 인쇄나 디자인처럼 곡선의 *시각적* 매끄러움이 중요한 응용에서 결정적이다.<br>
(a) has visible *kinks* at the knots; (b) is smooth but its curvature jumps at every knot; only (c) has smooth curvature throughout. The difference matters in any application that judges curves visually &mdash; CAD, typography, animation.


**한 절점에서 더 자세히 / Zoom in at one knot.** &nbsp; 함수값만 보면 PCHIP 와 자연 3차 스플라인이 비슷해 보일 수 있다 &mdash; 둘 다 매끄럽게 이어진다. 차이는 *미분* 에서 드러난다. 한 내부 절점 ($x = 2.0$) 주위로 확대하여 함수값과 1차 / 2차 미분을 함께 그려 보자.<br>
The differences between PCHIP and the natural cubic spline are subtle in $y(x)$ alone &mdash; both look smooth. The differences become obvious in the *derivatives*. Zoom in around one interior knot ($x = 2.0$) and plot $y$, $y'$, and $y''$ side-by-side.


In [ ]:
# 한 내부 절점 x = 2.0 주위로 확대 / zoom around one interior knot
x_zoom = np.linspace(1.0, 3.0, 400)

# 함수값 / function values
y_lin = np.interp(x_zoom, x_pts, y_pts)
y_pch = pchip(x_zoom)
y_spl = spline(x_zoom)

# 1차 미분 / first derivatives
def linear_slope(x_q, x_pts, y_pts):
    s = np.zeros_like(x_q)
    for i in range(len(x_pts) - 1):
        m = (y_pts[i+1] - y_pts[i]) / (x_pts[i+1] - x_pts[i])
        s[(x_q >= x_pts[i]) & (x_q < x_pts[i+1])] = m
    s[x_q >= x_pts[-1]] = m
    return s

dy_lin = linear_slope(x_zoom, x_pts, y_pts)
dy_pch = pchip(x_zoom, nu=1)
dy_spl = spline(x_zoom, nu=1)

# 2차 미분 / second derivatives
ddy_lin = np.zeros_like(x_zoom)              # 선형 안에서는 0 / zero inside each linear piece
ddy_pch = pchip(x_zoom, nu=2)
ddy_spl = spline(x_zoom, nu=2)

fig, axes = plt.subplots(3, 3, figsize=(12, 9), sharex=True)
labels = ['(a) piecewise linear ($C^0$)', '(b) PCHIP ($C^1$)', '(c) natural cubic spline ($C^2$)']

for ax, label in zip(axes[0], labels):
    ax.set_title(label)

# row 0: y(x)
for ax, y in zip(axes[0], (y_lin, y_pch, y_spl)):
    ax.plot(x_zoom, y)
    ax.plot(x_pts, y_pts, 'ko')
    ax.axvline(2.0, color='gray', linestyle=':', alpha=0.5)
    ax.set_ylabel('$y(x)$')
    ax.set_xlim(1.0, 3.0)
    ax.grid(True, alpha=0.3)

# row 1: y'(x)
for ax, dy in zip(axes[1], (dy_lin, dy_pch, dy_spl)):
    ax.plot(x_zoom, dy)
    ax.axvline(2.0, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_ylabel("$y'(x)$")
    ax.grid(True, alpha=0.3)

# row 2: y''(x)
for ax, ddy in zip(axes[2], (ddy_lin, ddy_pch, ddy_spl)):
    ax.plot(x_zoom, ddy)
    ax.axvline(2.0, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('$x$')
    ax.set_ylabel("$y''(x)$")
    ax.grid(True, alpha=0.3)

plt.suptitle('Zoom near knot $x = 2.0$', y=1.00)
plt.tight_layout()
plt.show()


행별로 무엇이 다른지 명확히 보인다.<br>
Reading row by row, the differences are stark:

* $y(x)$ &mdash; (a) 만 절점에서 꺾인다. (b), (c) 모두 매끄럽게 이어진다. / Only (a) has a kink at the knot; both (b) and (c) look smooth.
* $y'(x)$ &mdash; (a) 는 절점에서 *계단처럼 점프* 한다 (1차 미분 불연속). (b), (c) 모두 연속적이다. / The first derivative *jumps* at the knot for (a) (discontinuous slope). Both (b) and (c) are continuous.
* $y''(x)$ &mdash; (a) 는 (조각 안에서) $0$ 이지만 절점에서 정의되지 않고, (b) 도 절점에서 *불연속* 이다. 오직 (c) 만 절점을 매끄럽게 통과한다.<br>
  $y''$ is $0$ inside each linear piece for (a) (and undefined at the knot itself); for (b) it is *discontinuous* at the knot; only (c) passes through the knot continuously.

자연 3차 스플라인을 *시각적으로 매끄러운* 곡선으로 만드는 것은 이 $C^2$ 연속성이다 &mdash; 곡률 ($\propto y''$) 이 절점에서도 연속이라 눈으로는 절점이 어디인지 알 수 없다.<br>
The $C^2$ continuity is what makes the natural cubic spline *visually* smooth &mdash; curvature ($\propto y''$) is continuous through the knots, so the eye cannot tell where the knots are.


## 자연 3차 스플라인<br>The Natural Cubic Spline


**자연** 경계 조건은 양 끝에서 2차 미분을 0 으로 둔다.<br>
The **natural** boundary condition pins the second derivative to zero at both ends:

$$
S''(x_0) \;=\; S''(x_n) \;=\; 0.
$$

물리적으로, 무게추로 양 끝이 *자유롭게* 풀려 있는 얇은 나무 자가 만드는 곡선과 같다 &mdash; 자유로운 끝에서 굽힘 모멘트가 0 이고, 굽힘 모멘트는 곡률 ($\propto S''$) 에 비례한다.<br>
Physically, this is the curve a thin wooden ruler makes when its endpoints are *free* (not clamped) &mdash; bending moment, which is proportional to curvature $\propto S''$, vanishes at a free end.

미지수 $c_i = S''(x_i)/2$ 들에 대해 정리하면 *대각 항이 우세한* 삼중대각 (tridiagonal) 선형방정식이 된다. 작은 예제에서 확인해 보자.<br>
Solving for the second-derivative values $c_i = S''(x_i)/2$ gives a *tridiagonal*, diagonally-dominant linear system. Here is a small example.


In [ ]:
# 작은 예제: 자연 3차 스플라인이 푸는 삼중대각 시스템
# Small example: the tridiagonal system solved by a natural cubic spline
x_pts = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_pts = np.array([0.0, 0.8, 0.9, 0.1, -0.6])

n = len(x_pts) - 1                          # 조각 수 / number of pieces
h = np.diff(x_pts)                          # 조각 폭 / piece widths

# 내부 절점 i = 1 .. n-1 에 대한 식 (구간 폭이 다를 수 있으므로 일반형)
# For interior knots i = 1 .. n-1
A = np.zeros((n - 1, n - 1))
rhs = np.zeros(n - 1)
for i in range(1, n):
    if i > 1:
        A[i-1, i-2] = h[i-1]
    A[i-1, i-1] = 2.0 * (h[i-1] + h[i])
    if i < n - 1:
        A[i-1, i]   = h[i]
    rhs[i-1] = 6.0 * ((y_pts[i+1] - y_pts[i]) / h[i] -
                       (y_pts[i] - y_pts[i-1]) / h[i-1])

print('Tridiagonal coefficient matrix A =')
print(A)
print('Right-hand side b =', rhs)

m_interior = np.linalg.solve(A, rhs)         # 내부 절점에서의 S''
m = np.concatenate(([0.0], m_interior, [0.0]))   # 자연 BC: 양 끝에서 S'' = 0
print('S\'\'(x_i) =', m)


실제로는 직접 푸는 대신 `scipy.interpolate.CubicSpline` 을 사용한다 &mdash; 같은 시스템을 더 효율적이고 안정적으로 푼다.<br>
In practice we don&rsquo;t hand-solve this &mdash; `scipy.interpolate.CubicSpline` does the same thing, more efficiently and with better numerical care.


### `scipy.interpolate.CubicSpline`


**경계 조건 옵션** &mdash; SciPy 의 기본값은 `not-a-knot` 이지 자연 스플라인이 아니다. 자연 스플라인을 원한다면 명시적으로 지정해야 한다.<br>
**Important:** SciPy&rsquo;s default `bc_type` is `not-a-knot`, *not* the natural spline. If you want the natural spline, you must say so explicitly.


In [ ]:
x_pts = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_pts = np.array([0.0, 0.8, 0.9, 0.1, -0.6])

# 자연 / natural
S_nat = si.CubicSpline(x_pts, y_pts, bc_type='natural')
# 끝이 매듭이 아님 / not-a-knot (SciPy default)
S_nak = si.CubicSpline(x_pts, y_pts, bc_type='not-a-knot')
# 양 끝의 1차 미분을 0 으로 고정 / clamped (zero slopes at the ends)
S_clp = si.CubicSpline(x_pts, y_pts, bc_type=((1, 0.0), (1, 0.0)))

x_dense = np.linspace(x_pts[0], x_pts[-1], 400)

plt.figure(figsize=(8, 4.5))
plt.plot(x_dense, S_nat(x_dense), label='natural')
plt.plot(x_dense, S_nak(x_dense), label='not-a-knot (default)')
plt.plot(x_dense, S_clp(x_dense), label="clamped, $S'=0$")
plt.plot(x_pts, y_pts, 'ko')
plt.title('Effect of boundary conditions')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.legend(); plt.grid(True)
plt.show()


세 곡선 모두 자료점은 정확히 지나지만, 양 *끝* 에서의 거동이 다르다. 자료의 끝 부분에 대한 *사전 지식* 이 어떤 경계 조건을 고를지 결정한다.<br>
All three curves pass exactly through the data; they differ only in how they *behave at the boundaries*. Which boundary condition to pick depends on what you know about the data outside the interval.


## Runge 문제 다시 풀기<br>Revisiting the Runge Problem


앞 절의 룽게 함수에 등간격 절점에서 (a) 라그랑주 vs (b) 3차 스플라인 을 비교해 보자.<br>
Let&rsquo;s compare equispaced (a) Lagrange vs (b) cubic spline on the Runge function from the previous notebook.


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)


def lagrange_basis(x_nodes, j, x):
    x_nodes = np.asarray(x_nodes, dtype=float)
    x = np.asarray(x, dtype=float)
    result = np.ones_like(x)
    for i in range(len(x_nodes)):
        if i == j:
            continue
        result = result * (x - x_nodes[i]) / (x_nodes[j] - x_nodes[i])
    return result


def lagrange_interpolant(x_nodes, y_nodes, x):
    result = np.zeros_like(np.asarray(x, dtype=float))
    for j in range(len(x_nodes)):
        result = result + y_nodes[j] * lagrange_basis(x_nodes, j, x)
    return result


x_dense = np.linspace(-1.0, 1.0, 800)
y_true = runge(x_dense)

n = 15
x_nodes = np.linspace(-1.0, 1.0, n + 1)
y_nodes = runge(x_nodes)

y_lagr = lagrange_interpolant(x_nodes, y_nodes, x_dense)
y_spl  = si.CubicSpline(x_nodes, y_nodes, bc_type='natural')(x_dense)

plt.figure(figsize=(9, 5.5))
plt.plot(x_dense, y_true, 'k-', lw=2, label='$f$')
plt.plot(x_dense, y_lagr, 'r-',  label=f'Lagrange $p_{{{n}}}$')
plt.plot(x_dense, y_spl,  'b-',  label='Natural cubic spline')
plt.plot(x_nodes, y_nodes, 'ko')
plt.title('Lagrange vs cubic spline on Runge')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.legend(); plt.grid(True)
plt.show()


스플라인은 함수와 거의 일치하는 반면, 같은 *등간격* 절점에서 만든 15차 라그랑주 다항식은 양 끝에서 진동한다. 차수를 올리지 않고도 룽게 문제를 회피하는 것이 스플라인의 핵심 장점이다.<br>
The spline tracks $f$ closely, while the degree-15 Lagrange polynomial on the *same* equispaced nodes oscillates at the ends. Avoiding the Runge phenomenon *without* having to redesign the node placement is exactly the spline&rsquo;s superpower.


### 동적 탐색<br>Interactive Exploration


$n$ 을 키워 보자. 라그랑주는 점점 나빠지지만 스플라인은 점점 좋아진다.<br>
Sweep $n$ upward. Lagrange gets *worse*; the spline gets *better*.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_runge_compare(n):
    x_dense = np.linspace(-1.0, 1.0, 800)
    y_true = runge(x_dense)

    x_nodes = np.linspace(-1.0, 1.0, n + 1)
    y_nodes = runge(x_nodes)

    y_lagr = lagrange_interpolant(x_nodes, y_nodes, x_dense)
    y_spl  = si.CubicSpline(x_nodes, y_nodes, bc_type='natural')(x_dense)

    err_lagr = np.max(np.abs(y_lagr - y_true))
    err_spl  = np.max(np.abs(y_spl  - y_true))

    plt.figure(figsize=(8, 4.5))
    plt.plot(x_dense, y_true, 'k-', lw=2, label='$f$')
    plt.plot(x_dense, y_lagr, 'r-', label=f'Lagrange (err={err_lagr:.2g})')
    plt.plot(x_dense, y_spl,  'b-', label=f'Natural spline (err={err_spl:.2g})')
    plt.plot(x_nodes, y_nodes, 'ko')
    plt.title(f'n = {n}')
    plt.xlabel('$x$'); plt.ylabel('$y$')
    plt.legend(); plt.grid(True)
    plt.show()


if _ci:
    # 위젯 대신 첫 단계만 렌더 / Render only the first step instead of using widget
    plot_runge_compare(10)
else:
    interact(
        plot_runge_compare,
        n=IntSlider(min=4, max=30, step=1, value=10, description='n :'),
    );


## 어떤 방법을 쓸까<br>Choosing Between Lagrange and Spline


| 자료의 성격<br>Data character | 추천 방법<br>Recommended |
|---|---|
| 매끄러운 함수, 절점을 자유롭게 고를 수 있음<br>Smooth function, free choice of nodes | 라그랑주 + 체비셰프 절점 / Lagrange at Chebyshev nodes |
| 측정 자료, 절점은 고정, 부드럽게 잇고 싶음<br>Measured data, fixed nodes, want smoothness | 자연 3차 스플라인 / Natural cubic spline |
| 측정 자료, 양 끝의 1차 미분 정보가 있음<br>Measured data, end slopes known | 클램프드 3차 스플라인 / Clamped cubic spline |
| 자료가 단조 (monotone) 임이 보장되어야 함<br>Need monotonicity preserved | PCHIP / 1차 미분만 맞추는 Hermite |
| 단순히 자료점 사이 값만 빠르게 필요<br>Just need fast lookup between points | 선형 내삽 (`np.interp`) |

**일반적인 기본 선택은 자연 3차 스플라인이다.** 매끄러우면서 룽게 문제로부터 안전하다.<br>
**For most situations the default is the natural cubic spline.** It is smooth and robust to the Runge phenomenon.


## 연습 문제<br>Exercises


Try this 1: $\sin\theta^\circ$ 자료점을 $\theta = 0, 30, 60, \ldots, 360^\circ$ 에서 만든 후, 자연 3차 스플라인으로 $\theta = 1^\circ$ 마다 보간하시오. 정확한 $\sin$ 과 비교하여 최대 오차를 구하시오.<br>
Build $\sin\theta^\circ$ data on $\theta = 0, 30, 60, \ldots, 360^\circ$, fit a natural cubic spline, evaluate at $\theta = 1^\circ$, and report the maximum error against the true $\sin$.


Try this 2: 같은 자료에 `bc_type='not-a-knot'` 와 $\sin$ 의 양 끝 미분값을 사용한 클램프드 경계 조건을 비교하시오. 어느 것이 가장 정확한가?<br>
Repeat the previous exercise with `bc_type='not-a-knot'` and a clamped boundary using the true endpoint slopes of $\sin$. Which is most accurate?


Try this 3: 룽게 함수에 대해 $n = 8, 16, 32, 64$ 등간격 절점에서 자연 3차 스플라인의 최대 오차를 구하고, 수렴 차수를 추정하시오.<br>
For the Runge function, measure the maximum error of the natural cubic spline at $n = 8, 16, 32, 64$ equispaced nodes, and estimate the convergence rate.


## 참고문헌<br>References


* R. L. Burden, J. D. Faires, A. M. Burden, *Numerical Analysis*, 10th Ed., Cengage, 2016 (Sec. 3.5 *Cubic Spline Interpolation*).
* M. T. Heath, *Scientific Computing: An Introductory Survey*, 2nd Ed., McGraw-Hill, 2002 (Ch. 7 *Interpolation*).
* C. de Boor, *A Practical Guide to Splines*, Revised Ed., Springer, 2001.
* SciPy documentation, [`scipy.interpolate.CubicSpline`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html).


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");
